# Prepare the Data for Machine Learning Algorithms

Goal and anti-leakage rule

All transformers are fitted only on training data. Test data is transformed with fitted objects only


In [ ]:
import numpy as np
import pandas as pd

In [2]:
import os
from sklearn.datasets import fetch_openml

DATA_PATH = '../data/raw/credit_g.csv'

if not os.path.exists(DATA_PATH):
    raw = fetch_openml('credit-g', version=1, as_frame=True)
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    raw.frame.to_csv(DATA_PATH, index=False)

df = pd.read_csv(DATA_PATH, dtype={'class': 'category'})
df = df.rename(columns={'class': 'target'})


In [3]:
from sklearn.model_selection import train_test_split

# Since classes are 70/30, stratify preserve the ratio
X_train, X_test, y_train, y_test = train_test_split(df.drop('target', axis=1), df['target'], stratify=df['target'], random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(750, 20)
(250, 20)
(750,)
(250,)


## Preprocessing Strategy

This notebook prepares the data for modeling using one reproducible pipeline.

- `OrdinalEncoder` is used for columns that have a real order.
- `OneHotEncoder` is used for columns with no natural order.
- `log1p + StandardScaler` is used for skewed numeric columns.
- `StandardScaler` only is used for the rest of the numeric columns.

All transformations are fitted on the training set only and then applied to the test set.

In [4]:
ord_cols = {
    'checking_status': ['<0', '0<=X<200', '>=200', 'no checking'],
    'savings_status': ['<100', '100<=X<500', '500<=X<1000', '>=1000', 'no known savings'],
    'employment': ['unemployed', '<1', '1<=X<4', '4<=X<7', '>=7']
}

nom_cols = ['credit_history', 'purpose', 'housing', 'job', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'own_telephone', 'foreign_worker']

num_log_cols = ['credit_amount']
num_no_log_cols = X_train.select_dtypes(include='number').drop('credit_amount', axis=1).columns.tolist()

### Why these encoders and transformations?

We use the ordinal columns because they have a meaningful order. `OrdinalEncoder` turns the ordered categories into numbers while keeping that order.

The nominal columns do not have a real order. `OneHotEncoder` creates one binary column per category so the model does not assume any ranking.

`credit_amount` is positively skewed, so we apply `log1p`. This means `log(1 + x)`, which compresses large values and reduces the effect of outliers.

After that, `StandardScaler` puts numeric features on a similar scale. It makes the training process easier for models that are sensitive to feature magnitude.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.compose import ColumnTransformer

num_log_preprocessor = Pipeline(steps=[
    ('log', FunctionTransformer(np.log1p, validate=True)),
    ('scaler', StandardScaler())
])

num_no_log_preprocessor = Pipeline(steps=[
    ('scaler', StandardScaler())
])

ord_preprocessor = Pipeline(steps=[
    ('encoder', OrdinalEncoder(categories=[ord_cols[c] for c in ord_cols]))
])

nom_preprocessor = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [6]:
preprocessor = ColumnTransformer(transformers=[
    ('num_log', num_log_preprocessor, num_log_cols),
    ('num_no_log', num_no_log_preprocessor, num_no_log_cols),
    ('ord', ord_preprocessor, list(ord_cols.keys())),
    ('nom', nom_preprocessor, nom_cols)
], remainder='drop')

In [7]:
# preprocess data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [8]:
X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)
y_train = pd.Series(y_train)
y_test = pd.Series(y_test)

In [9]:
# Convert y_train and y_test to 1D series, then map good/bad to 0/1 once.
y_train = y_train.squeeze().map({"good": 0, "bad": 1})
y_test = y_test.squeeze().map({"good": 0, "bad": 1})

In [10]:
# save preprocessed data locally with directory creation
TRAIN_DIR = '../data/preprocessed/train'
TEST_DIR = '../data/preprocessed/test'

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

X_train.to_csv(f'{TRAIN_DIR}/X_train.csv', index=False)
X_test.to_csv(f'{TEST_DIR}/X_test.csv', index=False)
y_train.to_csv(f'{TRAIN_DIR}/y_train.csv', index=False)
y_test.to_csv(f'{TEST_DIR}/y_test.csv', index=False)

In [11]:
# save preprocessor
import joblib

joblib.dump(preprocessor, '../models/preprocessor.joblib')

['../models/preprocessor.joblib']

### Quality Check

The preprocessing step is valid if:

- the train and test matrices have matching feature columns,
- there are no missing values after transformation,
- the target split keeps the class ratio close to the original distribution,
- and the saved preprocessor can be reused later for inference.

If these checks pass, the data is ready for modeling in notebook 03.

In [12]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

print("Matching all columns: ", (X_train.columns == X_test.columns).all())

print("Nulls: ")
print(X_train.isnull().sum().sum(), X_test.isnull().sum().sum())

print("Class distribution:")
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

# Check saved preprocessor
preprocessor = None
preprocessor = joblib.load("../models/preprocessor.joblib")
print(preprocessor != None)

(750, 50)
(250, 50)
(750,)
(250,)
Matching all columns:  True
Nulls: 
0 0
Class distribution:
target
0    0.7
1    0.3
Name: proportion, dtype: float64
target
0    0.7
1    0.3
Name: proportion, dtype: float64
True


### Conclusion

The data is now split, encoded, scaled, and saved in a reusable format.

Target data is mapped good/bad to 0/1 and converted to 1D series.

Because the preprocessing is applied through a single fitted pipeline, the same logic can be reused during training and inference.

This is a good point to move to notebook 03 and start modeling.